# MSD / Diffusion Coefficient Analysis — Multi-Framework, Multi-Molecule, Multi-Loading

Processes `msd*.txt` files across:

```
BASE_DIR/
    UiO-66-UFF/
        CO2/  N2/  H2O/
            zero/            <- 0 loading (folder literally named "zero", or similar)
            0.5mmol/         <- any "<number>mmol" folder name, values not assumed
            1mmol/
            2.3mmol/
            ...
    MIL-101/
        CO2/  N2/  H2O/   (same pattern)
    UiO-66-MACE/
        CO2/  N2/  H2O/   (same pattern)
```

**Loading folder naming:** the exact mmol/g values aren't hard-coded anywhere — every
subfolder under a molecule folder is treated as its own loading point, and the numeric
value is parsed straight out of the folder name (`"1.5mmol"` → `1.5`, `"zero"` → `0.0`).
Whatever loadings actually exist on disk are the ones that get plotted.

**Per-framework settings:** UiO-66-UFF and MIL-101 use `real` units (0.5 fs/step);
UiO-66-MACE uses `metal` units (1 fs/step). Edit `FRAMEWORKS` below if this changes.

**Completeness rule:** an iteration is kept only if (1) its last recorded step reaches
≥90% of the max step seen across the group, and (2) it has ≥90% of the rows expected
between its own first and last step (tolerates a modest fraction of missing/dropped
lines). Corrupted rows (a line whose column count doesn't match the file's dominant
format — usually a crashed/truncated write) are dropped before any of this.

**Fitting, one plot per loading:**
- **D** is fit via the Einstein relation (straight line) over the **last 50%** of the
  trajectory, baseline-subtracted first. Its fit line is the only curve drawn on the plot.
- **α** is characterized over the **last 40%** of the trajectory: a straight line is
  tried first (α = 1, Fickian). If that line's R² is below 0.85, a power-law curve
  (MSD = A·t^α) is fit instead over the same window (only possible where the windowed
  data is strictly positive — a power law can't be fit through data that touches zero
  or goes negative, so in that case the straight-line result is kept). Whichever of the
  two has the higher R² is reported. **α's fit is never drawn on the plot** — only its
  value, R², and RMSE are reported (title + summary.csv).
- RMSE is reported alongside R² for both D and α.
- Iteration counts (`N = kept/total`) are **not** shown on the plot, but are still in
  `summary.csv`.

**Empty folders are skipped** with a printed message. The D-vs-loading plot for a given
framework/molecule is only produced if ≥2 numeric loading points exist for it.

**Per-combo cells:** Cell 7 defines the processing function; each framework/molecule
combo then has its own cell further down so you can run and inspect one at a time
without re-running the whole sweep. Re-running a single combo cell replaces just that
combo's rows, so it's safe to re-run without creating duplicates.

Just edit `BASE_DIR` in the Config cell below and run all cells.

## Cell 1 — Config
Only section you should need to edit.

In [ ]:
BASE_DIR = "."                # <-- point this at the folder containing the 3 framework dirs
OUT_DIR = "msd_analysis_output"

FRAMEWORKS = {
    # name must match the actual folder name on disk
    "UiO-66-UFF":  dict(timestep_fs=0.5),   # real units
    "MIL-101":     dict(timestep_fs=0.5),   # real units
    "UiO-66-MACE": dict(timestep_fs=1.0),   # metal units
}

MOLECULES = ["CO2", "N2", "H2O"]

MIN_COMPLETENESS = 0.90   # keep an iteration only if it reaches >=90% of the max step seen
FIT_START_FRAC = 0.5      # fit D (Einstein relation, straight line) over the LAST 50% of the trajectory
ALPHA_FIT_FRAC = 0.6      # fit alpha over the LAST 40% of the trajectory
ALPHA_R2_THRESHOLD = 0.85 # alpha: try a straight-line fit first; only fall back to a
                           # power-law curve fit if the straight line's R^2 is below this
N_BOOT = 200               # bootstrap resamples for D uncertainty

# Loading folder naming: "<number>mmol" for loaded points, "zero" (or similar) for empty.
# Exact mmol values are NOT assumed -- whatever number precedes "mmol" in the folder
# name is used directly. Add more zero-keywords here if your empty folders use a
# different name (e.g. "empty", "vacuum", "0mmol", bare "0").
LOADING_PATTERN_STR = r"(\d+(?:\.\d+)?)\s*mmol"
ZERO_KEYWORDS = {"zero", "empty", "blank", "vacuum", "0mmol"}

# Loadings to skip entirely (across every framework/molecule) -- e.g. 0.035mmol/g is the
# "2 molecules in the box" loading, excluded per request.
EXCLUDED_LOADINGS = {0.035}

# Per-(framework, molecule, loading_mmol_g) overrides for the D fit window (FIT_START_FRAC).
# Everything not listed here still uses the global FIT_START_FRAC.
FIT_START_FRAC_OVERRIDES = {
    ("MIL-101", "CO2", 0.021): 0.1,        # fit D over the LAST 90% instead of the last 50%
    ("MIL-101", "N2", 0.021): 0.1,
    ("MIL-101", "H2O", 0.021): 0.1,
    ("UiO-66-MACE", "CO2", 1.8): 0.1,
    ("UiO-66-MACE", "CO2", 0.6): 0.1,       # fit D over the LAST 90% instead of the last 50%
}

# Per-(framework, molecule, loading_mmol_g) MSD correction multipliers -- applied to
# mean_tot/std_tot/mat (all downstream: D fit, alpha fit, bootstrap error, plots) right
# after build_mean, before any fitting. Anything not listed here uses a scale of 1.0.
MSD_SCALE_OVERRIDES = {}

# Per-(framework, molecule, loading_mmol_g) extra trim for the D-vs-loading error bar:
# (n_low, n_high) = number of the LOWEST- and HIGHEST-D per-iteration estimates to drop
# from the bootstrap resampling pool, on top of the automatic MAD-based outlier flagging
# already done in flag_outlier_iterations. Use sparingly -- just to pull in an unusually
# wide error bar caused by a couple of extreme iterations, not a general cleaning knob.
D_ITER_TRIM_OVERRIDES = {
    ("MIL-101", "N2", 0.021): (3, 3),      # first loading point
    ("UiO-66-MACE", "N2", 0.6): (3, 3),    # first loading point
    ("UiO-66-UFF", "N2", 0.018): (10, 19), # first loading point -- heavy right tail; needs an
                                               # aggressive trim (31/56 iterations) to get the
                                               # error-bar span under 0.7 (scaled) as requested
    ("UiO-66-UFF", "N2", 5.0): (3, 3),     # last loading point
}

print("Config loaded.")


## Cell 2 — Imports

In [ ]:
import re
import csv
import warnings
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import linregress
from scipy.optimize import curve_fit

warnings.filterwarnings("ignore")
%matplotlib inline
plt.rcParams.update({"figure.dpi": 120, "font.size": 10})

BASE_DIR = Path(BASE_DIR)
OUT_DIR = Path(OUT_DIR)
OUT_DIR.mkdir(parents=True, exist_ok=True)
LOADING_PATTERN = re.compile(LOADING_PATTERN_STR, re.IGNORECASE)

COLOR_MAIN = "#2c3e50"
COLOR_FIT = "#f39c12"

print("Imports done.")


## Cell 3 — Folder / loading discovery

In [ ]:
def parse_loading_label(name: str):
    """Return a float loading (mmol/g) parsed from a folder name, or None if unknown."""
    m = LOADING_PATTERN.search(name)
    if m:
        return float(m.group(1))
    low = name.lower()
    if any(k in low for k in ZERO_KEYWORDS):
        return 0.0
    m2 = re.match(r"^(\d+(?:\.\d+)?)$", name.strip())
    if m2:
        return float(m2.group(1))
    return None


def is_excluded_loading(loading_value):
    """True if loading_value matches an entry in EXCLUDED_LOADINGS (float-safe compare)."""
    if loading_value is None:
        return False
    return any(abs(loading_value - ex) < 1e-9 for ex in EXCLUDED_LOADINGS)


def discover_loading_groups(mol_dir: Path):
    """Return list of (label, loading_value_or_None, [msd files]) for a molecule folder.
    Every subfolder found under mol_dir is treated as its own loading point --
    nothing about the actual mmol/g values is assumed in advance. Loadings listed in
    EXCLUDED_LOADINGS are skipped here so they never enter the pipeline at all (no row,
    no plot, no contribution to the D-vs-loading plot)."""
    groups = []
    subdirs = sorted([d for d in mol_dir.iterdir() if d.is_dir()])
    if subdirs:
        for d in subdirs:
            loading_value = parse_loading_label(d.name)
            if is_excluded_loading(loading_value):
                continue
            files = sorted(d.rglob("msd*.txt"))
            if not files:
                continue
            groups.append((d.name, loading_value, files))
    else:
        files = sorted(mol_dir.glob("msd*.txt"))
        if files:
            groups.append((mol_dir.name, None, files))
    return groups

print("Discovery functions defined.")


## Cell 4 — Parsing MSD files
Format per line: `iter step msd_x msd_y msd_z msd_total`

In [ ]:
def load_iterations(file_list):
    """Parse all msd*.txt files -> {global_iter_id: ndarray[(n,2)] columns=[step, msd_total]}.
    msd_total is always taken from the LAST column of each data line -- file formats vary
    (some have 6 columns: iter/step/x/y/z/total, others have 10 with two x/y/z/total
    decompositions), so column position isn't assumed, only that the last value is the
    total MSD to use.

    Within a single file, any line whose field count doesn't match that file's dominant
    (majority) field count is dropped as corrupted -- a crashed/interrupted job can merge
    or truncate a line mid-write, producing a stray row with an extra or missing column
    (and often a garbage step value). A single such row would otherwise poison the
    max-step reference used by filter_complete and wrongly mark every other iteration
    incomplete."""
    iters = {}
    gid = 0
    for fpath in file_list:
        raw_rows = []
        try:
            with open(fpath) as fh:
                for line in fh:
                    line = line.strip()
                    if not line or line.startswith("#"):
                        continue
                    parts = line.split()
                    if len(parts) < 3:
                        continue
                    try:
                        it = int(float(parts[0]))
                        step = float(parts[1])
                        msd_total = float(parts[-1])
                    except ValueError:
                        continue
                    raw_rows.append((it, step, msd_total, len(parts)))
        except OSError:
            continue

        if not raw_rows:
            continue

        field_counts = Counter(r[3] for r in raw_rows)
        majority_fields = field_counts.most_common(1)[0][0]
        raw_rows = [r for r in raw_rows if r[3] == majority_fields]

        local = {}
        for it, step, msd_total, _ in raw_rows:
            local.setdefault(it, []).append([step, msd_total])

        for lid, rows in sorted(local.items()):
            arr = np.array(sorted(rows, key=lambda r: r[0]), dtype=float)
            if len(arr) > 0:
                iters[gid] = arr
                gid += 1
    return iters


def filter_complete(iters, min_completeness):
    """Keep an iteration only if (1) its last recorded step reaches >= min_completeness
    of the max step seen across the group, AND (2) it has >= min_completeness of the rows
    expected between its own first and last step (tolerates a modest fraction of missing/
    dropped lines -- e.g. at the default 0.90, up to ~10% of lines can be missing and the
    iteration is still kept; the gaps are simply skipped when averaging across iterations
    in build_mean)."""
    if not iters:
        return {}
    max_step = max(arr[-1, 0] for arr in iters.values())
    if max_step <= 0:
        return {}

    kept = {}
    for k, arr in iters.items():
        if arr[-1, 0] / max_step < min_completeness:
            continue
        span = arr[-1, 0] - arr[0, 0]
        if span > 0:
            diffs = np.diff(arr[:, 0])
            diffs = diffs[diffs > 0]
            cadence = np.median(diffs) if len(diffs) else span
            n_expected = span / cadence + 1
            if n_expected > 0 and len(arr) / n_expected < min_completeness:
                continue
        kept[k] = arr
    return kept


def detect_print_every(all_steps):
    diffs = np.diff(np.unique(all_steps))
    diffs = diffs[diffs > 0]
    if len(diffs) == 0:
        return 1
    vals, counts = np.unique(diffs, return_counts=True)
    return int(vals[np.argmax(counts)])


def build_mean(kept_iters, timestep_fs):
    """Align kept iterations onto a common time grid; return time_ps, mean/std MSD_tot, raw matrix."""
    all_steps = np.concatenate([arr[:, 0] for arr in kept_iters.values()])
    print_every = detect_print_every(all_steps)
    max_step = int(all_steps.max())
    common_steps = np.arange(print_every, max_step + 1, print_every)
    time_ps = common_steps * timestep_fs * 1e-3
    step_to_idx = {s: i for i, s in enumerate(common_steps)}

    n_rec, n_it = len(common_steps), len(kept_iters)
    mat = np.full((n_rec, n_it), np.nan)
    for col, arr in enumerate(kept_iters.values()):
        for row in arr:
            idx = step_to_idx.get(int(row[0]))
            if idx is not None:
                mat[idx, col] = row[1]

    mean_tot = np.nanmean(mat, axis=1)
    std_tot = np.nanstd(mat, axis=1)
    return time_ps, mean_tot, std_tot, mat

print("Parsing functions defined.")


## Cell 5 — Fitting (Einstein relation + bootstrap error)

In [ ]:
def prepare_msd(time_ps, msd):
    """Drop NaNs and subtract the t=0 baseline from msd. Returns (t, m)."""
    valid = ~np.isnan(msd)
    t, m = time_ps[valid], msd[valid]
    if len(t) == 0:
        return t, m
    m = m - m[0]
    return t, m


def get_fit_start_frac(framework, molecule, loading_value, default=FIT_START_FRAC):
    """Look up a per-(framework, molecule, loading) override for the D fit window
    (FIT_START_FRAC_OVERRIDES); falls back to the global default when not listed."""
    if loading_value is None:
        return default
    for (fw, mol, val), frac in FIT_START_FRAC_OVERRIDES.items():
        if fw == framework and mol == molecule and abs(val - loading_value) < 1e-9:
            return frac
    return default


def get_msd_scale(framework, molecule, loading_value):
    """Look up a per-(framework, molecule, loading) MSD correction multiplier
    (MSD_SCALE_OVERRIDES); falls back to 1.0 (no change) when not listed."""
    if loading_value is None:
        return 1.0
    for (fw, mol, val), scale in MSD_SCALE_OVERRIDES.items():
        if fw == framework and mol == molecule and abs(val - loading_value) < 1e-9:
            return scale
    return 1.0


def get_extra_trim(framework, molecule, loading_value):
    """Look up a per-(framework, molecule, loading) extra outlier trim
    (D_ITER_TRIM_OVERRIDES) as (n_low, n_high); falls back to (0, 0) when not listed."""
    if loading_value is None:
        return (0, 0)
    for (fw, mol, val), trim in D_ITER_TRIM_OVERRIDES.items():
        if fw == framework and mol == molecule and abs(val - loading_value) < 1e-9:
            return trim
    return (0, 0)


def fit_D(t, m, ndim=3, start_frac=FIT_START_FRAC):
    """D = slope / (2*ndim) via a straight-line (Einstein relation) fit over the
    last (1-start_frac) of the window."""
    if len(t) < 6:
        return None
    start = int(len(t) * start_frac)
    if len(t) - start < 3:
        start = max(0, len(t) - 3)

    tt, mm = t[start:], m[start:]
    slope, intercept, r, _, _ = linregress(tt, mm)
    D = slope / (2 * ndim)
    fit_vals = intercept + slope * tt
    rmse = float(np.sqrt(np.mean((mm - fit_vals) ** 2)))

    return dict(slope=slope, intercept=intercept, R2=r ** 2, RMSE=rmse,
                D_A2ps=D, D_m2s=D * 1e-8, start=start, start_frac=start_frac)


def power_law(t, A, alpha):
    return A * np.power(t, alpha)


def fit_alpha(t, m, alpha_frac=ALPHA_FIT_FRAC, r2_threshold=ALPHA_R2_THRESHOLD):
    """Characterize the shape of MSD vs time over the last (1-alpha_frac) of the window.
    Tries a straight-line fit first (alpha = 1, Fickian). If that line's R^2 is below
    r2_threshold, also tries a power-law fit MSD = A*t**alpha over the same window --
    only possible where the windowed data is strictly positive, since a power law is
    undefined once the data touches zero or goes negative (in that case the straight
    line is kept as-is: "only fit the data" rather than force a power curve through it).
    Returns whichever of the two candidates has the higher R^2."""
    if len(t) < 6:
        return None
    start = int(len(t) * alpha_frac)
    if len(t) - start < 3:
        start = max(0, len(t) - 3)
    tt, mm = t[start:], m[start:]

    slope, intercept, r, _, _ = linregress(tt, mm)
    lin_fit_vals = intercept + slope * tt
    lin_R2 = r ** 2
    lin_rmse = float(np.sqrt(np.mean((mm - lin_fit_vals) ** 2)))
    best = dict(alpha=1.0, R2=lin_R2, RMSE=lin_rmse, fit_type="linear")

    if lin_R2 < r2_threshold:
        pos = (tt > 0) & (mm > 0)
        if pos.sum() > 6:
            tp, mp = tt[pos], mm[pos]
            try:
                p0 = [mp[-1] / (tp[-1] ** 1.0), 1.0]
                popt, _ = curve_fit(power_law, tp, mp, p0=p0, maxfev=10000)
                A_fit, alpha_fit = popt
                pl_fit_vals = power_law(tp, A_fit, alpha_fit)
                ss_res = np.sum((mp - pl_fit_vals) ** 2)
                ss_tot = np.sum((mp - np.mean(mp)) ** 2)
                pl_R2 = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan
                pl_rmse = float(np.sqrt(np.mean((mp - pl_fit_vals) ** 2)))
                if not np.isnan(pl_R2) and pl_R2 > best["R2"]:
                    best = dict(alpha=float(alpha_fit), R2=float(pl_R2),
                                RMSE=pl_rmse, fit_type="power_law")
            except Exception:
                pass

    return best


MAD_OUTLIER_THRESHOLD = 3.5   # modified z-score cutoff (Iglewicz & Hoya, 1993) for flagging an outlier iteration


def flag_outlier_iterations(time_ps, mat, ndim=3, start_frac=FIT_START_FRAC,
                             mad_threshold=MAD_OUTLIER_THRESHOLD, extra_trim=(0, 0)):
    """Fit D per-iteration and flag ones that deviate heavily from the group.
    Uses a MAD-based modified z-score (robust to the very outliers it's detecting,
    unlike a std-based z-score which the outlier itself would inflate).
    extra_trim=(n_low, n_high) additionally drops the n_low lowest- and n_high
    highest-D iterations (by value, among those the MAD pass kept) -- a manual,
    per-loading-point knob (D_ITER_TRIM_OVERRIDES) for pulling in an unusually wide
    error bar, layered on top of the automatic MAD screening.
    Returns a boolean keep-mask over mat's iteration axis (True = keep)."""
    n_it = mat.shape[1]
    D_iter = np.full(n_it, np.nan)
    for col in range(n_it):
        t, m = prepare_msd(time_ps, mat[:, col])
        res = fit_D(t, m, ndim=ndim, start_frac=start_frac)
        if res:
            D_iter[col] = res["D_m2s"]

    valid = ~np.isnan(D_iter)
    if valid.sum() < 3:
        return np.ones(n_it, dtype=bool)

    median = np.median(D_iter[valid])
    mad = np.median(np.abs(D_iter[valid] - median))
    if mad == 0:
        keep = np.ones(n_it, dtype=bool)
    else:
        mod_z = np.zeros(n_it)
        mod_z[valid] = 0.6745 * (D_iter[valid] - median) / mad
        keep = np.abs(mod_z) <= mad_threshold

    n_low, n_high = extra_trim
    if (n_low or n_high) and keep.sum() > (n_low + n_high):
        kept_idx = np.where(keep)[0]
        order = kept_idx[np.argsort(D_iter[kept_idx])]  # ascending by D
        drop = set(order[:n_low])
        if n_high:
            drop |= set(order[len(order) - n_high:])
        for idx in drop:
            keep[idx] = False

    return keep


def bootstrap_error(time_ps, mat, ndim=3, start_frac=FIT_START_FRAC, n_boot=N_BOOT,
                     mad_threshold=MAD_OUTLIER_THRESHOLD, extra_trim=(0, 0)):
    """Bootstrap the D uncertainty over iterations, excluding heavy-impact outlier
    iterations (per flag_outlier_iterations) from the resampling pool.
    extra_trim=(n_low, n_high): see flag_outlier_iterations.
    Returns (D_err, n_outliers)."""
    n_it = mat.shape[1]
    if n_it < 2:
        return np.nan, 0

    keep_mask = flag_outlier_iterations(time_ps, mat, ndim=ndim, start_frac=start_frac,
                                         mad_threshold=mad_threshold, extra_trim=extra_trim)
    n_outliers = int(n_it - keep_mask.sum())
    mat_clean = mat[:, keep_mask] if keep_mask.sum() >= 2 else mat
    if keep_mask.sum() < 2:
        n_outliers = 0

    n_clean = mat_clean.shape[1]
    D_list = []
    rng = np.random.RandomState(0)  # fixed seed: D_err is reproducible run to run
    for _ in range(n_boot):
        pick = rng.choice(n_clean, size=n_clean, replace=True)
        mean_col = np.nanmean(mat_clean[:, pick], axis=1)
        t, m = prepare_msd(time_ps, mean_col)
        res = fit_D(t, m, ndim=ndim, start_frac=start_frac)
        if res:
            D_list.append(res["D_m2s"])
    D_err = float(np.std(D_list)) if D_list else np.nan
    return D_err, n_outliers

print("Fitting functions defined.")


## Cell 6 — Plotting

In [ ]:
def plot_msd(framework, molecule, loading_label, loading_value,
             time_ps, mean_tot, res_D, res_alpha, out_dir):
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.plot(time_ps, mean_tot, color=COLOR_MAIN, lw=1.3, label="Data")

    if res_D:
        fit_line = res_D["intercept"] + res_D["slope"] * time_ps
        ax.plot(time_ps, fit_line, "--", color=COLOR_FIT, lw=1.5, label="Linear Fit")

    ax.set_xlabel("Time (ps)", fontweight="bold")
    ax.set_ylabel("MSD (Å²)", fontweight="bold")
    ax.legend(fontsize=8, loc="upper left", frameon=False)
    fig.tight_layout()

    safe_label = re.sub(r"[^A-Za-z0-9_.-]+", "_", loading_label)
    fname = out_dir / f"{framework}_{molecule}_{safe_label}_MSD.png"
    fig.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    return fname


_SUPERSCRIPT = str.maketrans("0123456789-", "\u2070\u00b9\u00b2\u00b3\u2074\u2075\u2076\u2077\u2078\u2079\u207b")


def choose_axis_power(values):
    """Pick the power-of-ten p such that values*10**p land in a readable
    ~[1, 1000) range, based on the largest-magnitude finite value."""
    finite = [abs(v) for v in values if np.isfinite(v) and v != 0]
    if not finite:
        return 0
    order = np.floor(np.log10(max(finite)))
    return -int(order)


def power_label(power, unit="m\u00b2/s"):
    """'D x 10^n (unit)' style label, e.g. power=8 -> 'D x 10\u2078 (m\u00b2/s)'."""
    if power == 0:
        return f"D ({unit})"
    return f"D \u00d7 10{str(power).translate(_SUPERSCRIPT)} ({unit})"


def plot_D_vs_loading(framework, molecule, points, out_dir):
    points = sorted(points, key=lambda p: p[0])
    x = [p[0] for p in points]
    y = [p[1] for p in points]
    yerr = [0 if np.isnan(p[2]) else p[2] for p in points]

    power = choose_axis_power(y)
    scale = 10.0 ** power
    y_scaled = [v * scale for v in y]
    yerr_scaled = [e * scale for e in yerr]

    fig, ax = plt.subplots(figsize=(6, 4.5))
    ax.errorbar(x, y_scaled, yerr=yerr_scaled, marker="o", capsize=4, color=COLOR_MAIN, lw=1.5)
    ax.set_xlabel("Loading (mmol/g)", fontweight="bold")
    ax.set_ylabel(power_label(power), fontweight="bold")
    ax.ticklabel_format(style="plain", axis="y")
    fig.tight_layout()

    fname = out_dir / f"{framework}_{molecule}_D_vs_loading.png"
    fig.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    return fname

print("Plotting functions defined.")


## Cell 7 — Per-group processing function
Defines a function that does the discover -> fit -> plot work for a single
framework/molecule combo. Run the cells further down (one per combo) to process
and inspect them individually.

In [ ]:
def process_group(framework, molecule):
    """Discover loading folders, fit D and alpha, and plot for a single framework/molecule
    combo. Returns (rows, d_points) for this combo -- does not mutate any global state."""
    rows = []
    d_points = []

    meta = FRAMEWORKS.get(framework)
    if meta is None:
        print(f"[skip] unknown framework: {framework}")
        return rows, d_points

    fw_dir = BASE_DIR / framework
    if not fw_dir.exists():
        print(f"[skip] framework folder not found: {fw_dir}")
        return rows, d_points

    ts_fs = meta["timestep_fs"]
    fw_out = OUT_DIR / framework
    fw_out.mkdir(parents=True, exist_ok=True)

    mol_dir = fw_dir / molecule
    if not mol_dir.exists() or not any(mol_dir.iterdir()):
        print(f"[skip] {framework}/{molecule}: folder missing or empty")
        return rows, d_points

    groups = discover_loading_groups(mol_dir)
    if not groups:
        print(f"[skip] {framework}/{molecule}: no msd*.txt files found")
        return rows, d_points

    for loading_label, loading_value, files in groups:
        iters = load_iterations(files)
        n_total = len(iters)
        tag = f"{framework}/{molecule}/{loading_label}"
        if n_total == 0:
            print(f"[skip] {tag}: unreadable or empty msd files")
            continue

        kept = filter_complete(iters, MIN_COMPLETENESS)
        n_kept = len(kept)
        print(f"{tag}: {n_kept}/{n_total} iterations ≥{int(MIN_COMPLETENESS*100)}% complete")

        if n_kept == 0:
            print(f"  [skip] no iterations pass the completeness threshold")
            continue

        time_ps, mean_tot, std_tot, mat = build_mean(kept, ts_fs)
        msd_scale = get_msd_scale(framework, molecule, loading_value)
        if msd_scale != 1.0:
            mean_tot = mean_tot * msd_scale
            std_tot = std_tot * msd_scale
            mat = mat * msd_scale
        t, m = prepare_msd(time_ps, mean_tot)
        d_start_frac = get_fit_start_frac(framework, molecule, loading_value)
        res_D = fit_D(t, m, ndim=3, start_frac=d_start_frac)
        if res_D is None:
            print(f"  [skip] not enough data points to fit D")
            continue
        res_alpha = fit_alpha(t, m)

        extra_trim = get_extra_trim(framework, molecule, loading_value)
        D_err, n_outliers = bootstrap_error(time_ps, mat, ndim=3, start_frac=d_start_frac,
                                             extra_trim=extra_trim)
        fname = plot_msd(framework, molecule, loading_label, loading_value,
                          time_ps, mean_tot, res_D, res_alpha, fw_out)
        outlier_note = f"  [{n_outliers} outlier iteration(s) excluded from error]" if n_outliers else ""
        alpha_str = f"{res_alpha['alpha']:.3f} ({res_alpha['fit_type']}, R2={res_alpha['R2']:.3f}, RMSE={res_alpha['RMSE']:.3g})" if res_alpha else "n/a"
        d_window_note = "" if d_start_frac == FIT_START_FRAC else f"  [D window overridden: last {int(round((1 - d_start_frac) * 100))}%]"
        scale_note = "" if msd_scale == 1.0 else f"  [MSD scaled x{msd_scale:.3g}]"
        print(f"  saved: {fname.name}  D = {res_D['D_m2s']:.3e} ± {D_err:.1e} m2/s  "
              f"R2 = {res_D['R2']:.3f}  RMSE = {res_D['RMSE']:.3g}  alpha = {alpha_str}{outlier_note}{d_window_note}{scale_note}")

        rows.append(dict(
            framework=framework, molecule=molecule, loading_label=loading_label,
            loading_mmol_g=loading_value, n_iter_kept=n_kept, n_iter_total=n_total,
            D_m2s=res_D['D_m2s'], D_err_m2s=D_err, n_outliers_excluded=n_outliers,
            D_R2=res_D['R2'], D_RMSE=res_D['RMSE'], D_fit_start_frac=d_start_frac,
            alpha=res_alpha['alpha'] if res_alpha else np.nan,
            alpha_fit_type=res_alpha['fit_type'] if res_alpha else None,
            alpha_R2=res_alpha['R2'] if res_alpha else np.nan,
            alpha_RMSE=res_alpha['RMSE'] if res_alpha else np.nan,
        ))

        if loading_value is not None:
            d_points.append((loading_value, res_D['D_m2s'], D_err, n_kept))

    return rows, d_points


def store_group_results(framework, molecule, rows, d_points):
    """Replace any previously stored rows/points for this framework/molecule combo.
    Makes it safe to re-run a single combo cell (e.g. after inspecting a plot) without
    accumulating duplicate rows in the summary."""
    global summary_rows
    summary_rows = [r for r in summary_rows
                    if not (r["framework"] == framework and r["molecule"] == molecule)]
    summary_rows.extend(rows)
    d_by_fw_mol[(framework, molecule)] = d_points


summary_rows = []
d_by_fw_mol = defaultdict(list)

print("Per-group processing function defined. Run each framework/molecule cell below to process and inspect it.")


### UiO-66-UFF — CO2

In [ ]:
rows, d_points = process_group("UiO-66-UFF", "CO2")
store_group_results("UiO-66-UFF", "CO2", rows, d_points)

### UiO-66-UFF — N2

In [ ]:
rows, d_points = process_group("UiO-66-UFF", "N2")
store_group_results("UiO-66-UFF", "N2", rows, d_points)

### UiO-66-UFF — H2O

In [ ]:
rows, d_points = process_group("UiO-66-UFF", "H2O")
store_group_results("UiO-66-UFF", "H2O", rows, d_points)

### MIL-101 — CO2

In [ ]:
rows, d_points = process_group("MIL-101", "CO2")
store_group_results("MIL-101", "CO2", rows, d_points)

### MIL-101 — N2

In [ ]:
rows, d_points = process_group("MIL-101", "N2")
store_group_results("MIL-101", "N2", rows, d_points)

### MIL-101 — H2O

In [ ]:
rows, d_points = process_group("MIL-101", "H2O")
store_group_results("MIL-101", "H2O", rows, d_points)

### UiO-66-MACE — CO2

In [ ]:
rows, d_points = process_group("UiO-66-MACE", "CO2")
store_group_results("UiO-66-MACE", "CO2", rows, d_points)

### UiO-66-MACE — N2

In [ ]:
rows, d_points = process_group("UiO-66-MACE", "N2")
store_group_results("UiO-66-MACE", "N2", rows, d_points)

### UiO-66-MACE — H2O

In [ ]:
rows, d_points = process_group("UiO-66-MACE", "H2O")
store_group_results("UiO-66-MACE", "H2O", rows, d_points)

## Cell 8 — D vs loading plots
Only produced where a framework/molecule has ≥2 distinct numeric loadings.

In [ ]:
for (framework, molecule), points in d_by_fw_mol.items():
    if len(points) < 2:
        print(f"[skip] {framework}/{molecule}: fewer than 2 loading points, skipping D-vs-loading plot")
        continue
    fw_out = OUT_DIR / framework
    fname = plot_D_vs_loading(framework, molecule, points, fw_out)
    print(f"saved D-vs-loading: {fname}")


## Cell 9 — Summary table

In [ ]:
if summary_rows:
    df = pd.DataFrame(summary_rows).sort_values(["framework", "molecule", "loading_mmol_g"])
    csv_path = OUT_DIR / "summary.csv"
    df.to_csv(csv_path, index=False)
    print(f"Summary table saved: {csv_path}")
    display(df)
else:
    df = pd.DataFrame()
    print("No results were produced -- check BASE_DIR and folder layout.")

print(f"\nAll plots in: {OUT_DIR.resolve()}")
